### Task C — Excess-Energy Flagging (Built on Task A)

**Idea:** 
Task A tells us how much power a slot *should* draw given its load and resources. Slots whose actual power is far above that expectation consume energy the load does not explain. 

In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from xgboost import XGBRegressor

X_train = pd.read_csv('../../data/X_train_taskA.csv')
y_train = pd.read_csv('../../data/y_train_taskA.csv').squeeze()
X_val = pd.read_csv('../../data/X_val_taskA.csv')
y_val = pd.read_csv('../../data/y_val_taskA.csv').squeeze()
X_test = pd.read_csv('../../data/X_test_taskA.csv')
y_test = pd.read_csv('../../data/y_test_taskA.csv').squeeze()

categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), numerical_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_cols)
])

model = Pipeline([
    ('preprocessor', preprocessor),
    ('reg', XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.02, subsample=0.9, colsample_bytree=0.6, random_state=42))
])

print("Training model on Train set to calculate validation thresholds...")
model.fit(X_train, y_train)
val_preds = model.predict(X_val)

res_val = y_val - val_preds
rel_val = res_val / val_preds

THR_ABS = res_val.mean() + 2 * res_val.std()
THR_REL = rel_val.mean() + 2 * rel_val.std()

print("Refitting model on Train+Val for final test evaluation...")
X_train_val = pd.concat([X_train, X_val], ignore_index=True)
y_train_val = pd.concat([y_train, y_val], ignore_index=True)
model.fit(X_train_val, y_train_val)

test_preds = model.predict(X_test)
res_test = y_test - test_preds
rel_test = res_test / test_preds

flag_abs = res_test > THR_ABS
flag_rel = rel_test > THR_REL

print(f"\n[I] ABSOLUTE rule: residual > {THR_ABS:.0f} W       -> flagged {flag_abs.sum()} of {len(y_test)} test slots ({(flag_abs.mean()*100):.1f}%)")
print(f"[II] RELATIVE rule: residual > {(THR_REL*100):.1f}% of expected power -> flagged {flag_rel.sum()} of {len(y_test)} test slots ({(flag_rel.mean()*100):.1f}%)")

/var/folders/t9/5qyb5pw554s_w_jt9c13myrh0000gn/T/ipykernel_77071/3591994002.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


Training model on Train set to calculate validation thresholds...
Refitting model on Train+Val for final test evaluation...

[I] ABSOLUTE rule: residual > 351 W       -> flagged 41 of 2250 test slots (1.8%)
[II] RELATIVE rule: residual > 25.4% of expected power -> flagged 28 of 2250 test slots (1.2%)


In [2]:
results_df = X_test.copy()
results_df['Total_Power_Watt'] = y_test
results_df['expected_W'] = test_preds
results_df['resid_W'] = res_test
results_df['resid_pct'] = rel_test * 100
results_df['flag_abs'] = flag_abs
results_df['flag_rel'] = flag_rel

results_df['kwh_per_mbps_x1e5'] = (results_df['Total_Power_Watt'] / np.maximum(results_df['Traffic_Volume_Mbps'], 1)) * 1e5

def group_nodes(n):
    if n <= 4: return '<=4 nodes'
    elif n <= 7: return '5-7 nodes'
    else: return '>=8 nodes'

results_df['node_grp'] = results_df['Active_Core_Nodes'].apply(group_nodes)

print("Flag rate by load group (a fair rule should not depend strongly on load):")
grp_summary = results_df.groupby('node_grp')[['flag_abs', 'flag_rel']].mean().round(3)
grp_summary['slots'] = results_df['node_grp'].value_counts()
display(grp_summary.reindex(['<=4 nodes', '5-7 nodes', '>=8 nodes']))

print("\nProfile of slots flagged by the RELATIVE rule vs the rest:")
profile_cols = {
    'Total_Power_Watt': 'mean', 
    'Traffic_Volume_Mbps': 'mean', 
    'Active_Core_Nodes': 'mean', 
    'CPU_Utilization_Pct': 'mean',
    'kwh_per_mbps_x1e5': 'mean'
}
profile_summary = results_df.groupby('flag_rel').agg(profile_cols).round(2)
profile_summary.insert(0, 'rows', results_df['flag_rel'].value_counts())
display(profile_summary)

print("\nTop 5 extreme excess slots (flagged by relative rule):")
worst_offenders = results_df[results_df['flag_rel']].sort_values('resid_pct', ascending=False)
cols_to_show = ['Traffic_Volume_Mbps', 'Active_Core_Nodes', 'CPU_Utilization_Pct', 'Total_Power_Watt', 'expected_W', 'resid_W', 'resid_pct']
display(worst_offenders[cols_to_show].head(5).round(1))

Flag rate by load group (a fair rule should not depend strongly on load):


,flag_abs,flag_rel,slots
node_grp,,,
<=4 nodes,0.005,0.025,644
5-7 nodes,0.011,0.008,1178
>=8 nodes,0.058,0.005,428



Profile of slots flagged by the RELATIVE rule vs the rest:


,rows,Total_Power_Watt,Traffic_Volume_Mbps,Active_Core_Nodes,CPU_Utilization_Pct,kwh_per_mbps_x1e5
flag_rel,,,,,,
False,2222,1411.87,3699.77,5.81,34.48,53979.76
True,28,1620.21,2270.99,4.61,30.70,106535.09



Top 5 extreme excess slots (flagged by relative rule):


,Traffic_Volume_Mbps,Active_Core_Nodes,CPU_Utilization_Pct,Total_Power_Watt,expected_W,resid_W,resid_pct
274,3517.3,5,42.8,1862.6,1409.599976,453.0,32.1
401,3407.8,7,39.2,1944.3,1476.900024,467.4,31.6
379,614.6,3,26.1,1427.4,1085.599976,341.8,31.5
964,535.7,4,22.8,1443.0,1108.599976,334.4,30.2
1669,1670.1,4,34.3,1554.0,1197.199951,356.8,29.8
